# Traditional Image Classification Pipeline

This notebook implements the traditional computer-vision pipeline for species
classification.

The main pipeline consists of:

1. Loading the shared dataset manifest.
2. Extracting SIFT local descriptors.
3. Constructing a visual vocabulary using MiniBatch K-Means.
4. Encoding each image as a Bag-of-Visual-Words histogram.
5. Training a Linear SVM classifier.
6. Training a Random Forest classifier.
7. Comparing validation and test performance.

This notebook does not use PyTorch. It depends only on NumPy, pandas,
Pillow, OpenCV, scikit-learn, matplotlib and joblib.

In [6]:
# --------------------------------------------------
# 1. Imports
# --------------------------------------------------

from pathlib import Path
import sys
import json
import time
import random

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

In [7]:
# --------------------------------------------------
# 2. Locate Project Root
# --------------------------------------------------

CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").is_dir():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").is_dir():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Please run this notebook from the project root "
        "or from the notebooks directory."
    )

SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIR}")

Project root: /Users/hanthony/Downloads/comp9517/COMP9517_Visionaries
Source directory: /Users/hanthony/Downloads/comp9517/COMP9517_Visionaries/src


In [ ]:
# --------------------------------------------------
# 3. Import Traditional Pipeline Functions
# --------------------------------------------------

from src.traditional.io import (
    load_manifest,
    get_class_names,
    get_split,
    load_rgb_image,
    iter_split,
    count_images_per_split,
    count_images_per_class,
    verify_image_paths,
)

from src.traditional.bovw import (
    sift_descriptors,
    sample_train_descriptors,
    build_codebook,
    encode_bovw,
    encode_split,
    build_classifier,
    scores_of,
)

from src.traditional.features import (
    FEATURES,
    extract_feature,
    extract_features,
)

print("Traditional pipeline modules imported successfully.")

In [ ]:
# --------------------------------------------------
# 4. Reproducibility
# --------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed: {SEED}")

In [ ]:
# --------------------------------------------------
# 5. Project Paths
# --------------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "traditional"
MODEL_DIR = OUTPUT_DIR / "models"
FEATURE_DIR = OUTPUT_DIR / "features"
RESULT_DIR = OUTPUT_DIR / "results"

MANIFEST_PATH = DATA_DIR / "manifest.csv"
IMAGE_ROOT = DATA_DIR

MODEL_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Manifest path: {MANIFEST_PATH}")
print(f"Image root:    {IMAGE_ROOT}")
print(f"Output folder: {OUTPUT_DIR}")

In [ ]:
# --------------------------------------------------
# 6. Experiment Configuration
# --------------------------------------------------

VOCAB_SIZE = 512
DESCRIPTORS_PER_IMAGE = 100
IMAGE_RESIZE = 256

BOVW_NORMALIZATION = "hellinger"

TRAIN_LIMIT = None
VALIDATION_LIMIT = None
TEST_LIMIT = None

SVM_NAME = "linear_svm"
RF_NAME = "random_forest"

EXPERIMENT_CONFIG = {
    "seed": SEED,
    "vocabulary_size": VOCAB_SIZE,
    "descriptors_per_image": DESCRIPTORS_PER_IMAGE,
    "image_resize": IMAGE_RESIZE,
    "bovw_normalization": BOVW_NORMALIZATION,
    "train_limit": TRAIN_LIMIT,
    "validation_limit": VALIDATION_LIMIT,
    "test_limit": TEST_LIMIT,
}

pd.Series(EXPERIMENT_CONFIG, name="value")